In [77]:
# ===== Robust Alpha Vantage loader + CHF risk functions =====

import os, io, time, json, hashlib, pathlib
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import time
load_dotenv()

FX_API = os.getenv("tiingo_api_key")
PRICE_API = os.getenv("eod_api_key")

# --- guard API key early ---
if (not FX_API or not isinstance(FX_API, str)) or (not PRICE_API or not isinstance(PRICE_API, str)):
    raise RuntimeError(
        "one or more api kays not found")

CACHE_DIR = pathlib.Path("tiingo_cache")
CACHE_DIR.mkdir(exist_ok=True)

today = time.strftime("%Y-%m-%d")
START = '2015-01-01'  # global start date for all fetches

In [87]:
def _cache_path(function: str, key: str) -> pathlib.Path:
    safe_key = key.replace("/", "_").replace(":", "_").replace(" ", "_")
    fname = f"{function}_{safe_key}.csv"
    return CACHE_DIR / fname

def _is_fresh(path: pathlib.Path, max_age: int) -> bool:
    if not path.exists():
        return False
    age_hours = (time.time() - path.stat().st_mtime) / 3600
    print(f"Cache file {path} age: {age_hours:.2f} hours (max age {max_age} hours)")
    result = age_hours <= max_age
    print(f'Is fresh: {result}')
    return result

def _looks_like_json(payload: bytes) -> bool:
    return payload.lstrip()[:1] in (b"{", b"[")

def fetch_csv_robust(url: str, function: str, key: str, max_age: int = 24) -> pd.DataFrame:
    """
        Robust CSV fetch with:
      - on-disk cache (TTL),
      - JSON throttle/error detection (does NOT overwrite cache),
      - atomic write on success.
        Returns a DataFrame with lowercase column names.
        """
    print(f"Fetching {function}:{key} ...")
    path = _cache_path(function, key)

    # if cache is fresh return it
    if _is_fresh(path, max_age):
        print(f"Using cached data for {function}:{key}")
        df = pd.read_csv(path,header=0, skiprows=[1], parse_dates=[0], index_col=0).sort_index()
        return df
    
    if function == "FX_DAILY":
        print('Pulling new data')
        headers = {
            'authorization': f'Token {FX_API}'
            }
        params = {
        'startDate': START,
        'endDate': today,
        'format': 'csv',
        'resampleFreq': '1day'
        }
        resp = requests.get(url,  params=params, headers=headers)

    elif function == "TIME_SERIES_DAILY_ADJUSTED":
        params_eq = {
        'from': START,  # EODHD uses from/to
        'to': today,
        'api_token': PRICE_API    
        }
        resp = requests.get(url, params=params_eq)
        
    resp.raise_for_status()
    raw = resp.content

    # Detect JSON throttle/error; do not poison cache
    if _looks_like_json(raw):
        # Try to show a concise message
        try:
            msg = json.loads(raw.decode("utf-8", errors="ignore"))
        except Exception:
            msg = {"body_head": raw[:200].decode("utf-8", errors="ignore")}
        raise RuntimeError(f"... returned JSON (throttle/error) for {function}:{key} -> {str(msg)[:180]}")

    # Parse CSV and normalize
    df = pd.read_csv(io.BytesIO(raw),header=0, skiprows=[1], parse_dates=[0], index_col=0).sort_index()
    print('will write new cache file')
    # Atomic-ish write
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        f.write(raw)
    os.replace(tmp, path)
    return df

def _pick_close_column(df: pd.DataFrame) -> pd.Series:
    # Prefer adjusted_close when available (better for split/dividend-adjusted series).
    # Fallback to close otherwise. Endpoint (DAILY vs DAILY_ADJUSTED) is chosen upstream
    if "adjClose" in df.columns:
        return df["adjClose"]
    raise ValueError("DataFrame does not contain 'adjClose' column.")

In [89]:
fxsym = 'chfusd'

url_fx=(f'https://api.tiingo.com/tiingo/fx/{fxsym}/prices')


df_test = fetch_csv_robust(url_fx, function="FX_DAILY", key=fxsym, max_age=100)

Fetching FX_DAILY:chfusd ...
Cache file tiingo_cache/FX_DAILY_chfusd.csv age: 21.28 hours (max age 100 hours)
Is fresh: True
Using cached data for FX_DAILY:chfusd


In [91]:

df_test.head()

,ticker,close,high,low,open
date,,,,,
2024-01-02 00:00:00+00:00,chfusd,1.176311,1.187980,1.175489,1.187613
2024-01-03 00:00:00+00:00,chfusd,1.177517,1.178134,1.168962,1.176305
2024-01-04 00:00:00+00:00,chfusd,1.176083,1.179593,1.172470,1.177510
2024-01-05 00:00:00+00:00,chfusd,1.176049,1.182669,1.165909,1.176083
2024-01-07 00:00:00+00:00,chfusd,1.175371,1.177881,1.174571,1.177881


In [ ]:


def build_returns_matrix_in_chf(
    holdings: list[dict],
    lookback_days: int = 252,
    use_adjusted: bool = True,
    max_age: int = 24,
    no_fx: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Build CHF daily returns matrix for the provided holdings.

        holdings: list of dicts with keys:
      - name: row/column label in outputs
      - symbol: Alpha Vantage symbol (e.g., 'SWDA.LON', 'IBM')
      - ccy: base currency of the asset price series (GBP/USD/EUR/CHF)
      - gbx: bool; if True, divide close by 100.0 (LSE pence)
      - position: number of shares held (float)

        Returns:
      rets_df: DataFrame of CHF log returns, T x N
      prices_df: DataFrame of CHF closes, T x N
      weights: Series aligned to columns in rets_df
        """

    # Pre-fetch FX once per currency (excluding CHF)
    fx_map: dict[str, pd.Series] = {}
    needed_ccy = sorted({h["ccy"].upper() for h in holdings if h["ccy"].upper() != "CHF"})
    for ccy in needed_ccy:

        url_fx = (f'https://api.tiingo.com/tiingo/fx/{ccy}chf/prices?token={FX_API}&startDate={START}&endDate={today}&format=csv&resampleFreq=1day')

        fx_df = fetch_csv_robust(url_fx, "FX_DAILY", f"{ccy}CHF", max_age=max_age)
        # fx_df["timestamp"] = pd.to_datetime(fx_df["timestamp"], utc=True)
        # fx_s = fx_df.sort_values("timestamp").set_index("timestamp")["close"].rename(f"{ccy}CHF")
        fx_df["timestamp"] = pd.to_datetime(fx_df['date'], utc=True)
        fx_df = fx_df.sort_values("timestamp").set_index("timestamp")
        fx_s = fx_df.loc[:, "close"].rename(f"{ccy}CHF")  # explicit, Series with name
        fx_map[ccy] = fx_s

    # Build CHF close series per asset
    chf_close = {}
    for h in holdings:
        name  = h["name"]
        sym   = h["symbol"]
        ccy   = h["ccy"].upper()
        gbx   = bool(h.get("gbx", False))

        fn = "TIME_SERIES_DAILY_ADJUSTED" 

        url_px = (f'https://api.tiingo.com/tiingo/fx/{sym}/prices?startDate=2024-1-1&endDate={today}&format=csv&resampleFreq=1day')
    
        px_df = fetch_csv_robust(url_px, fn, sym, max_age=max_age)
        px_df["timestamp"] = pd.to_datetime(px_df["timestamp"], utc=True)
        px_df = px_df.sort_values("timestamp").set_index("timestamp")

        close_local = _pick_close_column(px_df)
        if gbx:
            # LSE pence → GBP. (Does not change returns, only magnitude.)
            close_local = close_local / 100.0

        if (ccy == "CHF") or no_fx:
            close_chf = close_local.rename(name)
        else:
            fx = fx_map[ccy]
            df = pd.concat([close_local.rename("p"), fx.rename("x")], axis=1).dropna()
            close_chf = (df["p"] * df["x"]).rename(name)

        chf_close[name] = close_chf
        # print(f'chf_close["{name}"] tail:\n{chf_close[name].tail()}')
        # print the last value in 'chf_close[name]
    # print(f'Last value: {chf_close}')



    # Align on common dates and restrict to lookback window
    prices_df = pd.DataFrame(chf_close).dropna()
    prices_df = prices_df.tail(lookback_days)
    rets_df = np.log(prices_df / prices_df.shift(1)).dropna()

    if prices_df.shape[0] < (lookback_days * 0.9):
        raise ValueError(
            f"After alignment only {prices_df.shape[0]} rows remain "
            f"(expected {lookback_days}). Data source may not have full history."
    )
    if rets_df.isna().any().any():
        raise ValueError("NaNs remained in returns after shift/drop; check data alignment.")
    if (prices_df <= 0).any().any():
        raise ValueError("Non-positive prices encountered; check source data.")

    values = {}
    for h in holdings:
        name = h['name']
        last_price = chf_close[name].iloc[-1]
        values[name] = h['position'] * last_price
        # print(h['name'], h['position'], f'{last_price:.2f}', f'{values[name]:.2f}')

    # sum the values in values
    total_val = sum(values.values())
    print(f"Total portfolio value (CHF): {total_val:.2f}")


    # Weights (CHF)
    # total_val = sum(float(h["value_chf"]) for h in holdings)
    w = pd.Series({h["name"]: float(values[h["name"]]) / total_val for h in holdings})
    w = w.reindex(rets_df.columns).fillna(0.0)
    if not np.isclose(w.sum(), 1.0, atol=1e-6):
        raise ValueError(f"Weights must sum to 1. Got {w.sum():.6f}"
                         "check postions input in holdings.")

    return rets_df, prices_df, w

def portfolio_risk(rets_df: pd.DataFrame, weights: pd.Series) -> dict:
    """
    Compute annualized vols, correlation, covariance, portfolio vol,
    marginal risk contribution (MRC), and percent risk contribution (PRC).
    """
    # Annualized stats
    cov_daily = rets_df.cov()
    cov_annual = cov_daily * 252.0
    vol_ann = rets_df.std() * np.sqrt(252.0)
    corr = rets_df.corr()

    # Align weights
    w = weights.reindex(rets_df.columns).astype(float)
    Sigma_w = cov_annual @ w
    port_var = float(w @ Sigma_w)
    port_vol = float(np.sqrt(port_var)) if port_var > 0 else 0.0

    # Contributions
    mrc = Sigma_w / port_vol if port_vol > 0 else Sigma_w * 0.0
    prc = (w * Sigma_w) / port_var if port_var > 0 else w * 0.0

    summary = pd.DataFrame({
        "Weight": w,
        "Vol_1Y_CHF": vol_ann,
        "MRC": mrc,           # ∂σ_p/∂w_i (absolute marginal contribution)
        "PRC_%": prc * 100.0  # percent contribution to total variance (sums ~100%)
    }).sort_values("PRC_%", ascending=False)

    return {
        "port_vol": port_vol,
        "cov_annual": cov_annual,
        "corr": corr,
        "vol_ann": vol_ann,
        "mrc": mrc,
        "prc": prc,
        "summary": summary,
    }

In [ ]:
sym = 'CHFUSD'
url_px=(f'https://api.tiingo.com/tiingo/fx/{sym}/prices?token={API_KEY}&startDate=2024-1-1&endDate={today}&format=csv&resampleFreq=1day')

df_test = fetch_csv_robust

In [ ]:
def _cache_path(function: str, key: str) -> pathlib.Path:
    safe_key = key.replace("/", "_").replace(":", "_").replace(" ", "_")
    fname = f"{function}_{safe_key}.csv"
    return CACHE_DIR / fname

def _is_fresh(path: pathlib.Path, max_age: int) -> bool:
    if not path.exists():
        return False
    age_hours = (time.time() - path.stat().st_mtime) / 3600
    # print(f"Cache file {path} age: {age_hours:.2f} hours (TTL {max_age} hours)")
    return age_hours <= max_age

def _looks_like_json(payload: bytes) -> bool:
    return payload.lstrip()[:1] in (b"{", b"[")

def fetch_csv_robust(url: str, function: str, key: str, max_age: int = 24) -> pd.DataFrame:
    """
        Robust CSV fetch with:
      - on-disk cache (TTL),
      - JSON throttle/error detection (does NOT overwrite cache),
      - schema validation against REQ[function],
      - atomic write on success.
        Returns a DataFrame with lowercase column names.
        """
    path = _cache_path(function, key)

    # if cache is stale fetch new data
    if _is_fresh(path, max_age):
        print(f"Using cached data for {function}:{key}")
        # Parse cached CSV with the same options to preserve DatetimeIndex
        df = pd.read_csv(path, header=0, skiprows=[1], parse_dates=[0], index_col=0).sort_index()
        return df

    # Pull from network
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    raw = resp.content

    # Detect JSON throttle/error; do not poison cache
    if _looks_like_json(raw):
        # Try to show a concise message
        try:
            msg = json.loads(raw.decode("utf-8", errors="ignore"))
        except Exception:
            msg = {"body_head": raw[:200].decode("utf-8", errors="ignore")}
        raise RuntimeError(f"... returned JSON (throttle/error) for {function}:{key} -> {str(msg)[:180]}")

    # Parse CSV and normalize
    df = pd.read_csv(io.BytesIO(raw),header=0, skiprows=[1], parse_dates=[0], index_col=0).sort_index()

    # Atomic-ish write
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        f.write(raw)
    os.replace(tmp, path)
    return df

def _pick_close_column(df: pd.DataFrame) -> pd.Series:
    # Prefer adjusted_close when available (better for split/dividend-adjusted series).
    # Fallback to close otherwise. Endpoint (DAILY vs DAILY_ADJUSTED) is chosen upstream
    if "adjClose" in df.columns:
        return df["adjClose"]
    raise ValueError("DataFrame does not contain 'adjClose' column.")